In [ ]:
import pandas as pd
import glob
import os
from tqdm import tqdm
import numpy as np

PATH_INPUT_1 = "source_data_1.xlsx"
PATH_INPUT_2 = "source_data_2.xlsx"
PATH_INPUT_3 = "lookup_table.parquet"
PATH_DATA_MAIN = "main_database.parquet"
BASE_OUTPUT_DIR = "FINAL_REPORTS"

df_1 = pd.read_excel(PATH_INPUT_2, sheet_name="Source_Sheet")
TARGET_VALUES = ["ALPHA", "BETA", "GAMMA"]
df_filtered = df_1[df_1['CATEGORY_COL'].isin(TARGET_VALUES)]

df_2 = pd.read_parquet(PATH_INPUT_3)

df_1['ID_KEY'] = df_1['ID_KEY'].astype(str).str.strip().str.upper()

df_3 = pd.read_parquet(PATH_DATA_MAIN)
df_3['DATE'] = pd.to_datetime(df_3['DATE'], dayfirst=True, errors='coerce')
df_3 = df_3.dropna(subset=['DATE'])
df_3['PERIOD_UNIT'] = df_3['DATE'].dt.isocalendar().week

df_2 = df_2.apply(lambda x: x.str.upper().str.strip() if x.dtype == "object" else x)
df_3.columns = [col.upper() for col in df_3.columns]
df_3 = df_3.apply(lambda x: x.str.upper().str.strip() if x.dtype == "object" else x)

df_3 = df_3.fillna("N.D.")
df_3 = df_3.replace(r'^\s*$', "N.D.", regex=True)
df_2 = df_2.fillna("N.D.")
df_2 = df_2.replace(r'^\s*$', "N.D.", regex=True)

UNIQUE_PERIODS = sorted(df_3['PERIOD_UNIT'].unique())

for period in UNIQUE_PERIODS:
    df_current_period = df_3[df_3['PERIOD_UNIT'] == period].copy()
    df_current_period['ID_LINK'] = df_current_period['ID_LINK'].astype(str).str.strip().str.upper()

    df_1_2 = pd.merge(df_current_period, df_2, left_on='JOIN_KEY_A', right_on='JOIN_KEY_B', how='inner')

    df_1_2['FINAL_DESC'] = np.where(
        (df_1_2['FINAL_DESC'].isna()) | (df_1_2['FINAL_DESC'].astype(str).str.strip().isin(['', 'N.D.'])), 
        df_1_2['SECONDARY_DESC'], 
        df_1_2['FINAL_DESC']
    )

    df_subset_main = df_1_2[df_1_2['FINAL_DESC'].isin(['ALPHA', 'BETA'])].copy()
    
    df_1_2_3 = pd.merge(
        df_subset_main,
        df_1[['ID_KEY', 'GROUP_LEVEL_1', 'GROUP_LEVEL_2']], 
        left_on='ID_LINK',
        right_on='ID_KEY',
        how='inner'
    ).rename(columns={'GROUP_LEVEL_2': 'LOC_REFERENCE'})

    unique_groups = df_1_2_3['GROUP_LEVEL_1'].unique()
    for group in unique_groups:
        group_name_clean = str(group).upper().strip()
        group_folder = group_name_clean.replace(' ', '_')
        df_group = df_1_2_3[df_1_2_3['GROUP_LEVEL_1'] == group].copy()

        if group_name_clean != "EXCEPTION_CASE":
            df_group = df_group[df_group['LOC_TAG'].astype(str).str.strip().str.upper() == 
                                df_group['LOC_REFERENCE'].astype(str).str.strip().str.upper()].copy()
        
        if df_group.empty: continue

        export_path = os.path.join(BASE_OUTPUT_DIR, group_name_clean, "SUB_FOLDER_A", f"Report_Primary_{group_folder}", "2026")
        os.makedirs(export_path, exist_ok=True)
        
        file_name = f"2026_P{period}_Report_Primary_{group_folder}.xlsx"
        full_file_path = os.path.join(export_path, file_name)
        
        with pd.ExcelWriter(full_file_path, engine='xlsxwriter') as writer:
            for sub_id in df_group['ID_LINK'].unique():
                df_sub = df_group[df_group['ID_LINK'] == sub_id].copy()
                df_sub = df_sub.sort_values(by='DATE')

                df_pivot = df_sub.groupby(['DATE', 'OPERATOR', 'ID_LINK', 'SUB_DESC', 'LOC_REFERENCE', 'FINAL_DESC']).size().unstack(fill_value=0).reset_index()
                
                for col in ['ALPHA', 'BETA']:
                    if col not in df_pivot.columns: df_pivot[col] = 0

                df_pivot['TOTAL_SUM'] = df_pivot['ALPHA'] + df_pivot['BETA']
                df_final_sheet = df_pivot[['DATE', 'OPERATOR', 'ID_LINK', 'SUB_DESC', 'LOC_REFERENCE', 'ALPHA', 'BETA', 'TOTAL_SUM']]
                
                sum_a = df_final_sheet['ALPHA'].sum()
                sum_b = df_final_sheet['BETA'].sum()
                sum_total = df_final_sheet['TOTAL_SUM'].sum()
                
                totals_row = pd.DataFrame({
                    'DATE': ['TOTAL'], 'OPERATOR': [''], 'ID_LINK': [''], 'SUB_DESC': [''], 
                    'LOC_REFERENCE': [''], 'ALPHA': [sum_a], 'BETA': [sum_b], 'TOTAL_SUM': [sum_total]
                })
                
                df_final_sheet = pd.concat([df_final_sheet, totals_row], ignore_index=True)
                df_final_sheet['DATE'] = df_final_sheet['DATE'].apply(lambda x: x.strftime('%Y-%m-%d') if isinstance(x, pd.Timestamp) else x)

                sheet_name = str(sub_id)[:31].replace(':', '_')
                df_final_sheet.to_excel(writer, sheet_name=sheet_name, index=False)
                
                workbook = writer.book
                worksheet = writer.sheets[sheet_name]
                (max_row, max_col) = df_final_sheet.shape

                header_fmt = workbook.add_format({'bold': True, 'font_color': '#FFFFFF', 'bg_color': '#2F5597', 'border': 1})
                
                column_settings = [{'header': column} for column in df_final_sheet.columns]
                worksheet.add_table(0, 0, max_row - 1, max_col - 1, {'columns': column_settings, 'style': 'Table Style Medium 9'})

                for i, col in enumerate(df_final_sheet.columns):
                    column_len = max(df_final_sheet[col].astype(str).len().max(), len(col)) + 3
                    worksheet.set_column(i, i, column_len)

for period in UNIQUE_PERIODS:
    df_current_period = df_3[df_3['PERIOD_UNIT'] == period].copy()
    df_1_2_alt = pd.merge(df_current_period, df_2, left_on='JOIN_KEY_A', right_on='JOIN_KEY_B', how='inner')
    
    df_alt = df_1_2_alt[~df_1_2_alt['FINAL_DESC'].isin(['ALPHA', 'BETA'])].copy()
    
    if df_alt.empty: continue
    
    df_1_2_3_alt = pd.merge(df_alt, df_1[['ID_KEY', 'GROUP_LEVEL_1', 'GROUP_LEVEL_2']], left_on='ID_LINK', right_on='ID_KEY', how='inner')

    for group in df_1_2_3_alt['GROUP_LEVEL_1'].unique():
        group_name_clean = str(group).upper().strip()
        df_group_alt = df_1_2_3_alt[df_1_2_3_alt['GROUP_LEVEL_1'] == group].copy()

        export_path_alt = os.path.join(BASE_OUTPUT_DIR, group_name_clean, "SUB_FOLDER_B", f"Report_Secondary_{group_name_clean}", "2026")
        os.makedirs(export_path_alt, exist_ok=True)
        
        file_name_alt = f"2026_P{period}_Report_Secondary_{group_name_clean}.xlsx"
        with pd.ExcelWriter(os.path.join(export_path_alt, file_name_alt), engine='xlsxwriter') as writer:
            for sub_id in df_group_alt['ID_LINK'].unique():
                df_sub_alt = df_group_alt[df_group_alt['ID_LINK'] == sub_id].copy()
                df_pivot_alt = df_sub_alt.groupby(['DATE', 'OPERATOR', 'ID_LINK', 'SUB_DESC', 'FINAL_DESC']).size().unstack(fill_value=0).reset_index()
                
                df_pivot_alt.to_excel(writer, sheet_name=str(sub_id)[:31], index=False)
                # [Logica di formattazione simile alla precedente applicata qui]

for period in UNIQUE_PERIODS:
    df_current_period = df_3[df_3['PERIOD_UNIT'] == period].copy()
    df_1_2_comp = pd.merge(df_current_period, df_2, left_on='JOIN_KEY_A', right_on='JOIN_KEY_B', how='inner')

    df_1_2_3_comp = pd.merge(
        df_1_2_comp, 
        df_1[['ID_KEY', 'GROUP_LEVEL_1', 'GROUP_LEVEL_2']].rename(columns={'GROUP_LEVEL_2': 'AUTH_LOC'}), 
        left_on='ID_LINK', right_on='ID_KEY', how='inner'
    )

    for group in df_1_2_3_comp['GROUP_LEVEL_1'].unique():
        df_group_comp = df_1_2_3_comp[df_1_2_3_comp['GROUP_LEVEL_1'] == group].copy()
        
        mask_discrepancy = df_group_comp.apply(lambda x: str(x['LOC_TAG']) not in str(x['AUTH_LOC']), axis=1)
        df_discrepancy = df_group_comp[mask_discrepancy].copy()
        
        if df_discrepancy.empty: continue

        export_path_disc = os.path.join(BASE_OUTPUT_DIR, str(group), "SUB_FOLDER_C", "Discrepancy_Reports", "2026")
        os.makedirs(export_path_disc, exist_ok=True)
import pandas as pd
import numpy as np
import os

# --- SEZIONE 1: ELABORAZIONE DETTAGLIO CATEGORIE ---

with pd.ExcelWriter(percorso_output_1, engine='xlsxwriter') as writer:
    fogli_scritti = 0
    
    for item_id in df_2['KEY_ID'].unique():
        df_sub_1 = df_2[df_2['KEY_ID'] == item_id].copy()
        
        if df_sub_1.empty:
            continue
        
        df_sub_1 = df_sub_1.sort_values(by=['FIELD_DATE', 'FIELD_TIME'])

        df_final_1 = df_sub_1[[
            'TIMESTAMP_1', 
            'TIMESTAMP_2', 
            'OPERATOR_ID', 
            'ID_READ',
            'KEY_ID', 
            'KEY_DESCRIPTION', 
            'ITEM_TYPE',                 
            'LOCATION_REF'
        ]].rename(columns={'LOCATION_REF': 'LOCATION_MATCH'})

        sheet_name_clean = str(item_id)[:31].replace(':', '_').replace('[','').replace(']','')
        df_final_1.to_excel(writer, sheet_name=sheet_name_clean, index=False)
        
        workbook  = writer.book
        worksheet = writer.sheets[sheet_name_clean]
        (max_row, max_col) = df_final_1.shape 

        fmt_val_1 = workbook.add_format({'bg_color': '#C6EFCE', 'font_color': '#006100'})
        fmt_val_2 = workbook.add_format({'bg_color': '#FFEB9C', 'font_color': '#9C5700'})

        idx_target = df_final_1.columns.get_loc('ITEM_TYPE')
        col_letter = chr(65 + idx_target)

        worksheet.conditional_format(1, 0, max_row, max_col - 1, {
            'type':     'formula',
            'criteria': f'OR(${col_letter}2="TYPE_A", ${col_letter}2="TYPE_B")',
            'format':   fmt_val_1
        })

        worksheet.conditional_format(1, 0, max_row, max_col - 1, {
            'type':     'formula',
            'criteria': f'AND(${col_letter}2<>"TYPE_A", ${col_letter}2<>"TYPE_B")',
            'format':   fmt_val_2
        })

        column_settings = [{'header': column} for column in df_final_1.columns]
        worksheet.add_table(0, 0, max_row, max_col - 1, {
            'columns': column_settings,
            'style': 'Table Style Light 1'
        })
        
        for i, col in enumerate(df_final_1.columns):
            max_len = max(df_final_1[col].astype(str).map(len).max(), len(col)) + 3
            worksheet.set_column(i, i, min(max_len, 50))
        
        fogli_scritti += 1

# --- SEZIONE 2: ANALISI ID NON TROVATI ---

for period_val in list_periods:
    df_3 = df_1[df_1['PERIOD'] == period_val].copy().dropna(subset=['OPERATOR_ID'])
    df_3 = df_3[df_3['OPERATOR_ID'].astype(str).str.upper() != 'N.D.']
    
    df_4 = pd.merge(df_3, df_lookup_1, left_on='ID_READ', right_on='ID_REF', how='left')
    df_5 = df_4[df_4['ID_REF'].isna()].copy()
    
    if df_5.empty:
        continue

    df_6 = pd.merge(
        df_5, 
        df_lookup_2[['KEY_ID_REF', 'UNIT_ID']], 
        left_on='KEY_ID', 
        right_on='KEY_ID_REF', 
        how='inner'
    )

    for unit in df_6['UNIT_ID'].unique():
        unit_clean = str(unit).upper().strip()
        unit_folder = unit_clean.replace(' ', '_')
        
        path_output_2 = os.path.join(base_dir, unit_clean, "REPORTS", f"REP_MISSING_{unit_folder}", "2026")
        os.makedirs(path_output_2, exist_ok=True)
        
        file_name_2 = f"2026_P{period_val}_MISSING_IDS_{unit_folder}.xlsx"
        full_path_2 = os.path.join(path_output_2, file_name_2)
        
        df_7 = df_6[df_6['UNIT_ID'] == unit].copy()

        with pd.ExcelWriter(full_path_2, engine='xlsxwriter') as writer:
            for key in df_7['KEY_ID'].unique():
                df_8 = df_7[df_7['KEY_ID'] == key].copy()
                df_8 = df_8.sort_values(by='FIELD_DATE')
                
                df_9 = df_8.groupby(['FIELD_DATE', 'OPERATOR_ID', 'KEY_ID']).size().reset_index(name='COUNT_MISSING')
                
                df_9['DATE_LABEL'] = df_9['FIELD_DATE'].dt.strftime('%A %d-%m-%Y').replace({
                    'Monday': 'Lunedì', 'Tuesday': 'Martedì', 'Wednesday': 'Mercoledì', 
                    'Thursday': 'Giovedì', 'Friday': 'Venerdì', 'Saturday': 'Sabato', 'Sunday': 'Domenica'
                }, regex=True)
                
                df_10 = df_9[['DATE_LABEL', 'OPERATOR_ID', 'KEY_ID', 'COUNT_MISSING']]
                
                df_total_row = pd.DataFrame({
                    'DATE_LABEL': ['TOTAL_PERIOD'], 
                    'OPERATOR_ID': ['-'], 
                    'KEY_ID': ['-'], 
                    'COUNT_MISSING': [df_10['COUNT_MISSING'].sum()]
                })
                
                df_final_2 = pd.concat([df_10, df_total_row], ignore_index=True).rename(columns={'DATE_LABEL': 'DATE'})
                
                sheet_name_clean = str(key)[:31].replace(':', '_')
                df_final_2.to_excel(writer, sheet_name=sheet_name_clean, index=False)

                workbook  = writer.book
                worksheet = writer.sheets[sheet_name_clean]
                (max_row, max_col) = df_final_2.shape

                fmt_total = workbook.add_format({
                    'bold': True, 'font_color': '#FFFFFF', 'bg_color': '#C65911',
                    'top': 1, 'bottom': 6, 'align': 'left'
                })
                
                column_settings = [{'header': column} for column in df_final_2.columns]
                worksheet.add_table(0, 0, max_row - 1, max_col - 1, {
                    'columns': column_settings,
                    'style': 'Table Style Medium 14'
                })
                
                for col_num, value in enumerate(df_final_2.iloc[-1]):
                    worksheet.write(max_row, col_num, value, fmt_total)

                for i, col in enumerate(df_final_2.columns):
                    max_len = max(df_final_2[col].astype(str).map(len).max(), len(col)) + 3
                    worksheet.set_column(i, i, min(max_len, 50))

# --- SEZIONE 3: ELENCO ANALITICO ---

for period_val in list_periods:
    df_11 = df_1[df_1['PERIOD'] == period_val].copy().dropna(subset=['OPERATOR_ID'])
    df_12 = pd.merge(df_11, df_lookup_1, left_on='ID_READ', right_on='ID_REF', how='left')
    df_13 = df_12[df_12['ID_REF'].isna()].copy()
    df_13 = df_13[df_13['ID_READ'].astype(str).str.upper() != 'N.D.']

    if df_13.empty:
        continue

    df_14 = pd.merge(
        df_13, 
        df_lookup_2[['KEY_ID_REF', 'UNIT_ID']], 
        left_on='KEY_ID', 
        right_on='KEY_ID_REF', 
        how='inner'
    )

    for unit in df_14['UNIT_ID'].unique():
        unit_clean = str(unit).upper().strip()
        unit_folder = unit_clean.replace(' ', '_')
        
        path_output_3 = os.path.join("ROOT", "DATA", "REPORTS", unit_clean, f"DET_MISSING_{unit_folder}")
        os.makedirs(path_output_3, exist_ok=True)
        
        file_name_3 = f"2026_P{period_val}_ANALYTICAL_LIST_{unit_folder}.xlsx"
        full_path_3 = os.path.join(path_output_3, file_name_3)
        
        df_15 = df_14[df_14['UNIT_ID'] == unit].copy()
        df_export_1 = df_15[['ID_READ', 'FIELD_DATE', 'KEY_ID']].copy()
        df_export_1 = df_export_1.sort_values(by=['FIELD_DATE', 'KEY_ID'])

        if not df_export_1.empty:
            df_export_1['FIELD_DATE'] = pd.to_datetime(df_export_1['FIELD_DATE']).dt.strftime('%d/%m/%Y')

        with pd.ExcelWriter(full_path_3, engine='xlsxwriter') as writer:
            sheet_name = "Analytical_Detail"
            df_export_1.to_excel(writer, sheet_name=sheet_name, index=False)
            
            workbook  = writer.book
            worksheet = writer.sheets[sheet_name]
            (max_row, max_col) = df_export_1.shape

            column_settings = [{'header': column} for column in df_export_1.columns]
            worksheet.add_table(0, 0, max_row, max_col - 1, {
                'columns': column_settings,
                'style': 'Table Style Medium 1' 
            })

            for i, col in enumerate(df_export_1.columns):
                max_len = max(df_export_1[col].astype(str).map(len).max(), len(col)) + 3
                worksheet.set_column(i, i, min(max_len, 50))

# --- SEZIONE 4: RIEPILOGO PER UNITÀ OPERATIVA ---

valid_validation_set = set(zip(df_lookup_2['SUB_UNIT'].astype(str).str.upper().str.strip(), 
                               df_lookup_2['KEY_ID_REF'].astype(str).str.upper().str.strip()))

unit_mapping = dict(zip(df_lookup_2['SUB_UNIT'].astype(str).str.upper().str.strip(), 
                        df_lookup_2['UNIT_ID'].astype(str).strip().str.upper()))

correction_map = {'OLD_VAL_1': 'NEW_VAL_1', 'OLD_VAL_2': 'NEW_VAL_2'}

for period_val in list_periods:
    df_16 = df_1[df_1['PERIOD'] == period_val].copy()
    df_17 = pd.merge(df_16, df_lookup_1, left_on='ID_READ', right_on='ID_REF', how='inner')

    df_17['LOCATION_REF'] = df_17['LOCATION_REF'].astype(str).str.upper().str.strip().replace(correction_map)

    df_17['ITEM_TYPE'] = np.where(
        (df_17['ITEM_TYPE'].isna()) | (df_17['ITEM_TYPE'].astype(str).str.strip() == ''), 
        df_17['ALT_TYPE'], 
        df_17['ITEM_TYPE']
    )

    df_17 = df_17[df_17['ITEM_TYPE'].str.upper().isin(['TARGET_VAL_A', 'TARGET_VAL_B'])].copy()
    
    if df_17.empty: continue

    results_by_unit = {}
    for sub_unit, df_group in df_17.groupby('LOCATION_REF'):
        unit_up = unit_mapping.get(sub_unit, "UNDEFINED_UNIT")
        if unit_up not in results_by_unit: 
            results_by_unit[unit_up] = {}
        results_by_unit[unit_up][sub_unit] = df_group

    for unit, groups in results_by_unit.items():
        unit_folder = unit.replace(' ', '_')
        path_output_4 = os.path.join(base_dir, unit, "REPORTS", f"SUMMARY_UNIT_{unit_folder}", "2026")
        os.makedirs(path_output_4, exist_ok=True)
        
        file_name_4 = f"2026_P{period_val}_SUMMARY_{unit_folder}.xlsx"
        full_path_4 = os.path.join(path_output_4, file_name_4)
        
        with pd.ExcelWriter(full_path_4, engine='xlsxwriter') as writer:
            fmt_ok = writer.book.add_format({'bg_color': '#C6EFCE', 'font_color': '#006100', 'border': 1})
            fmt_error = writer.book.add_format({'bg_color': '#FFC7CE', 'font_color': '#9C0006', 'border': 1})
            
            for sub_unit_name, df_data in groups.items():
                df_data = df_data.sort_values(by='FIELD_DATE')
                df_summary = df_data.groupby(['FIELD_DATE', 'KEY_ID']).size().reset_index(name='COUNT_READS')
                
                df_summary['DATE_STR'] = df_summary['FIELD_DATE'].dt.strftime('%A %d-%m-%Y').replace({
                    'Monday': 'Lunedì', 'Tuesday': 'Martedì', 'Wednesday': 'Mercoledì', 
                    'Thursday': 'Giovedì', 'Friday': 'Venerdì', 'Saturday': 'Sabato', 'Sunday': 'Domenica'
                }, regex=True)
                
                df_summary['UNIT_REF'] = unit
                df_final_3 = df_summary[['DATE_STR', 'KEY_ID', 'UNIT_REF', 'COUNT_READS']].rename(columns={'DATE_STR': 'DATE'})
                
                sheet_name = str(sub_unit_name)[:31].replace(':', '_')
                df_final_3.to_excel(writer, sheet_name=sheet_name, index=False)
                
                worksheet = writer.sheets[sheet_name]
                for row_idx, row_val in df_final_3.iterrows():
                    is_valid = (str(sub_unit_name).upper().strip(), str(row_val['KEY_ID']).upper().strip()) in valid_validation_set
                    format_to_apply = fmt_ok if is_valid else fmt_error
                    
                    for col_idx, value in enumerate(row_val):
                        worksheet.write(row_idx + 1, col_idx, value, format_to_apply)

                (max_row, max_col) = df_final_3.shape
                worksheet.add_table(0, 0, max_row, max_col - 1, {
                    'columns': [{'header': c} for c in df_final_3.columns],
                    'style': 'Table Style Light 1'
                })

                for i, col in enumerate(df_final_3.columns):
                    max_len = max(df_final_3[col].astype(str).map(len).max(), len(col)) + 3
                    worksheet.set_column(i, i, min(max_len, 40))

missing_units = df_17[~df_17['LOCATION_REF'].str.upper().str.strip().isin(unit_mapping.keys())]['LOCATION_REF'].unique()       

In [ ]:
Riassunto del Codice
Questo script Python implementa una pipeline di Data Processing e Reporting Automatizzato per la gestione di flussi logistici o di rilevazione (come scansioni ID o log operativi). Le principali funzionalità includono:

Integrazione e Pulizia Dati: Combina diversi set di dati (sorgenti grezze, dizionari di lookup e tabelle di anagrafica) tramite operazioni di merge (join) e filtraggio avanzato per isolare anomalie o record specifici.

Generazione Report Multi-livello: Produce automaticamente file Excel complessi organizzati per unità operativa e sottocategorie. Ogni report include fogli multipli dinamici basati sui dati elaborati.

Automazione Excel Avanzata: Utilizza la libreria xlsxwriter per:

Creare Tabelle Excel native.

Applicare Formattazione Condizionale basata su formule (es. evidenziare errori o tipologie specifiche).

Gestire il layout (auto-fit delle colonne, stili personalizzati, righe di totale calcolate on-the-fly).

Validazione Incrociata (Business Logic): Implementa controlli di coerenza tra le diverse tabelle, evidenziando graficamente (tramite colori nel report) se un'attività è stata registrata correttamente rispetto ai parametri previsti.

Gestione File System: Gestisce in autonomia la creazione di strutture di cartelle nidificate organizzate per anno, unità e tipologia di report, garantendo un'archiviazione ordinata degli output.